# Practice 3. Metrics for imbalanced fault data

Industrial AI, KBTU. Goes with Lecture 5 and `inclass_work_5.ipynb`. **1.25 points.**

**Data: SECOM**, a semiconductor manufacturing line. 1567 wafer lots, 590 anonymous signals,
104 lots failed the final test. The notebook downloads the file from Kaggle by itself.

<https://www.kaggle.com/datasets/paresh2047/uci-semcom>

**The process.** SECOM comes from a semiconductor fabrication line (McCann and Johnston, UCI
Machine Learning Repository, 2008). Each row is one wafer lot passing a stage of the line.
590 signals are collected from sensors and measurement points along the way. At the end the
lot is tested: it passes or fails.

**What is and is not known.** The signals are **anonymised**: no names, no units, no process
knowledge. You cannot build physics features as in In-class work 4. The practice tests how far
generic models and honest metrics get when nobody tells you what the sensors are.

**The columns after loading:** `Time` (when the lot was measured), `s000` … `s589` (the
signals) and `Pass/Fail`.

**The costs in this practice.** An extra inspection of a lot costs **1** unit. A failed lot sent
on to the next stages costs **20** units: it takes line time and materials and is scrapped at
the end. These two numbers are `C_FP` and `C_FN` in the code.

### How the tasks are built

Each task says what to write: the functions, their arguments and what they return, and sometimes
a variable with a result. How you write them is up to you, as long as you use the method the task
names. Under your cell sits a cell of **tests**. Run it after your code: it either prints
`all tests passed` or stops at the first check that failed and says what is wrong. The tests check
what your code does on small cases with known answers, not how it is written.

These tests are public. When you hand in, the same tests run again together with hidden ones, on
other inputs, and the share of tests passed gives the points for the code. Code that only fits the
public cases loses those points.

Cells marked *Common ground* are written for you. Run them as they are: they fix the label, the
split and the two models, so that everybody's numbers can be compared.

The metrics are yours: the confusion counts, precision, recall and F1, the ROC curve and its area
by pairs, the PR curve and average precision, the cost curve. Write them with NumPy from the
formulas. `sklearn.metrics` is there to check your result, not to compute it.

### What you hand in

Two files:

1. this notebook, run top to bottom, named `Surname_Name_practice_3.ipynb`;
2. `practice_3_answers.md` with your answers to the questions under each task, named
   `Surname_Name_practice_3_answers.md`.

Every answer needs the numbers your notebook printed. A number the notebook does not print scores
zero, and so does a number printed with nothing said about it.

| Task | 1 | 2 | 3 | 4 | 5 | 6 | 7 | total |
|---|---|---|---|---|---|---|---|---|
| code (tests) | 0.05 | 0.08 | 0.08 | 0.10 | 0.10 | 0.12 | 0.08 | 0.61 |
| answers | 0.05 | 0.07 | 0.07 | 0.10 | 0.10 | 0.13 | 0.12 | 0.64 |

### The rules

- **The test set is touched only to measure.** Thresholds and choices between models are made
  on the development part, with cross-validation. A threshold chosen on the test set scores
  zero for the answers of Task 6.
- Every precision, recall or F1 is reported **with its threshold**.
- Do not rename the functions and variables the tests use, and do not edit the test cells.
- Before handing in, restart the kernel and run all cells; the whole notebook runs in about two
  minutes on a laptop. Then check your answers against the new output.
- Figures need axis labels and a title that says what to look at.
- Stuck? `inclass_work_5.ipynb` does every step on AI4I.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

import sklearn

BLUE, ORANGE, RED, GREEN, PURPLE, GREY = (
    "#2a6f8e", "#e07b39", "#c0392b", "#4a8f5a", "#9b3f9b", "#9aa0a6")

mpl.rcParams.update({
    "figure.dpi": 110,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "#cccccc",
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": "#e6e6e6",
    "grid.linewidth": 0.8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.titlelocation": "left",
    "axes.labelsize": 10,
    "font.size": 10,
    "font.family": ["Segoe UI", "DejaVu Sans", "sans-serif"],
    "legend.frameon": False,
    "lines.linewidth": 1.6,
})

pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

KAGGLE = {
    'secom': 'paresh2047/uci-semcom',
}


def dataset_folder(name):
    """The folder with the files of a data set: the Kaggle copy (downloaded once, then cached),
    or data/<name>/ of the course repo if Kaggle is out of reach."""
    if KAGGLE[name]:
        try:
            import contextlib, io, logging, kagglehub
            logging.getLogger("kagglehub").setLevel(logging.ERROR)
            with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
                folder = Path(kagglehub.dataset_download(KAGGLE[name]))
            print(f"{name}: Kaggle {KAGGLE[name]}")
            return folder
        except Exception as exc:
            print(f"Kaggle unavailable ({type(exc).__name__}), looking for data/{name}/")
    for root in [Path.cwd(), *Path.cwd().parents]:
        if (root / "data" / name).is_dir():
            return root / "data" / name
    raise FileNotFoundError(f"no data/{name}/ here or above; run data/fetch.py in the course repo")

print(" | ".join(f"{m} {sys.modules[m].__version__}" for m in
                 ("numpy", "pandas", "matplotlib", "scipy", "sklearn", "torch") if m in sys.modules))

numpy 2.5.5 | pandas 3.1.1 | matplotlib 3.11.2 | scipy 1.18.1 | sklearn 1.9.1


In [2]:
def load_secom():
    df = pd.read_csv(next(dataset_folder("secom").rglob("uci-secom.csv")))
    # sensor columns come as "0" ... "589"; give them names that are easier to type
    return df.rename(columns={c: f"s{int(c):03d}" for c in df.columns if c.isdigit()})


secom = load_secom()
print(secom.shape)
secom.iloc[:5, :6]

secom: Kaggle paresh2047/uci-semcom
(1567, 592)


,Time,s000,s001,s002,s003,s004
0,2008-07-19 11:55:00,"3,030.930","2,564.000","2,187.733","1,411.127",1.360
1,2008-07-19 12:32:00,"3,095.780","2,465.140","2,230.422","1,463.661",0.829
2,2008-07-19 13:17:00,"2,932.610","2,559.940","2,186.411","1,698.017",1.510
3,2008-07-19 14:43:00,"2,988.720","2,479.900","2,199.033",909.793,1.320
4,2008-07-19 15:22:00,"3,032.240","2,502.870","2,233.367","1,326.520",1.533


---

## Task 1. What is in the file  
*0.10 points: 0.05 for the code (tests), 0.05 for the answers*

Take inventory before you model anything. Two things are worth knowing about the label and
the signals: how rare a failure is, and how much of the table is empty or constant.

Then the clock. `pd.to_datetime(secom["Time"])` parses the column. Check that it increases
down the file. It does not: find the rows where time jumps backwards, look at the dates just
before and after each jump, and work out what went wrong when the file was written. A date like
2008-01-08 between 31 July and 1 August is a hint. Repair the column so that it gives the true
time of every lot.

**Write**

- `missing_share`: a `float`, the share of missing values among all values of the 590 signals;
- `n_mostly_missing`: an `int`, the number of signals with more than half of their values missing;
- `n_constant`: an `int`, the number of signals with one distinct value (ignoring NaN);
- `backward_jumps(t)`: for a `pd.Series` of timestamps, a 1-D integer array of the **positions**
  $i$ (0, 1, 2, … whatever the index) where `t` at $i$ is earlier than at $i - 1$. Equal times are
  not a jump;
- `lot_time`: a `pd.Series` of timestamps, the repaired time of every lot, with the index of
  `secom`;
- `weekly`: a `pd.DataFrame` from `lot_time` grouped by week (`resample("W")`), with columns
  `"failed"` (failed lots), `"lots"` (all lots) and `"rate"` (their ratio).

In [ ]:
missing_share = ...
n_mostly_missing = ...
n_constant = ...


def backward_jumps(t):
    ...


lot_time = ...
weekly = ...

In [ ]:
# Tests for Task 1 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_jumps_back_in_time():
    days = ["2008-01-01", "2008-01-03", "2008-01-02", "2008-01-05", "2007-12-31"]
    t = pd.Series(pd.to_datetime(days))
    got = backward_jumps(t)
    assert got is not None, "backward_jumps returns the positions"
    assert list(got) == [2, 4], f"time goes back at positions 2 and 4, got {got}"


test_jumps_back_in_time()


def test_inventory_types():
    assert isinstance(missing_share, float), "missing_share: a float between 0 and 1"
    assert 0 <= missing_share <= 1, "missing_share: a float between 0 and 1"
    for v in (n_mostly_missing, n_constant):
        assert isinstance(v, (int, np.integer)) and 0 <= v <= 590, \
            "n_mostly_missing, n_constant: counts of signals"


test_inventory_types()


def test_repaired_time_increases():
    assert isinstance(lot_time, pd.Series), "lot_time: one timestamp per lot"
    assert len(lot_time) == len(secom), "lot_time: one timestamp per lot"
    assert pd.api.types.is_datetime64_any_dtype(lot_time), "lot_time holds timestamps"
    assert lot_time.is_monotonic_increasing, "the repaired time must not go backwards"


test_repaired_time_increases()


def test_weekly_table():
    assert isinstance(weekly, pd.DataFrame), "weekly: a DataFrame"
    assert {"failed", "lots", "rate"} <= set(weekly.columns), "weekly: failed, lots, rate"
    assert ((weekly["rate"] >= 0) & (weekly["rate"] <= 1)).all(), "rate is a share"


test_weekly_table()


print("all tests passed")

**Questions** (answer them in `practice_3_answers.md`, Task 1, with the numbers your code printed):

1. Which value of `Pass/Fail` means a failed lot, and how many lots of each are there? How accurate is a model that says every lot passes?
2. Report the share of missing values, the number of signals more than half empty and the number of constant signals.
3. Where does `Time` jump backwards? Quote two jumps with the dates before and after, say what went wrong, how you repaired it and what the true first and last dates are.
4. Plot or print the failed lots per week. Are failures spread evenly over the three months?

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

## Common ground for Tasks 2–7

Run this cell as it is. It fixes the label (1 = failed), the split and the two models, so that
everybody's numbers can be compared. Use `make_logreg()` and `make_forest()` whenever a task
says "the logistic regression" or "the forest", and `CV` for cross-validation inside the
development part.

Both models are trained with balanced class weights, the usual first fix for imbalance. Keep
in mind what In-class work 5, Part 5.3 showed about such scores.

In [6]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

SENSORS = [c for c in secom.columns if c.startswith("s")]
X = secom[SENSORS]
y = (secom["Pass/Fail"] == 1).astype(int).to_numpy()        # 1 = the lot FAILED

# the same split for everybody: 70 % development, 30 % test, the same failure rate in both
split = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
X_dev, X_test, y_dev, y_test = split
CV = StratifiedKFold(5, shuffle=True, random_state=0)    # folds of the development set


def make_logreg():
    logreg = LogisticRegression(C=0.01, class_weight="balanced", max_iter=5000)
    return make_pipeline(SimpleImputer(strategy="median"), VarianceThreshold(),
                         StandardScaler(), logreg)


def make_forest():
    forest = RandomForestClassifier(500, min_samples_leaf=3,
                                    class_weight="balanced_subsample",
                                    random_state=0, n_jobs=-1)
    return make_pipeline(SimpleImputer(strategy="median"), forest)


C_FP, C_FN = 1, 20      # an extra inspection of a lot; a failed lot sent downstream

print(f"development: {len(y_dev)} lots, {y_dev.sum()} failed | "
      f"test: {len(y_test)} lots, {y_test.sum()} failed")

development: 1096 lots, 73 failed | test: 471 lots, 31 failed


---

## Task 2. The confusion matrix  
*0.15 points: 0.08 for the code (tests), 0.07 for the answers*

With 1 = failed lot:

| | predicted pass (0) | predicted fail (1) |
|---|---|---|
| **really passed (0)** | TN | FP, a false alarm |
| **really failed (1)** | FN, a missed failure | TP |

Each cell counts the lots where two conditions hold at once, so each is the sum of a product of
two boolean masks.

**Write**

- `confusion_counts(y_true, y_pred)`: for two arrays of 0/1 (or booleans) of the same length, the
  tuple `(tn, fp, fn, tp)` of Python `int`s. No `sklearn` inside;
- `logreg`, `forest`: the logistic regression and the forest fitted on the development part;
- `s_lr`, `s_rf`: their scores on the test part, `predict_proba(X_test)[:, 1]`, as 1-D arrays.
  Every later task uses them.

In [ ]:
def confusion_counts(y_true, y_pred):
    ...


logreg = ...
forest = ...
s_lr = ...
s_rf = ...

In [ ]:
# Tests for Task 2 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_six_lots_by_hand():
    got = confusion_counts([0, 0, 0, 1, 1, 0], [0, 1, 0, 1, 0, 1])
    assert got == (2, 2, 1, 1), f"expected (tn, fp, fn, tp) = (2, 2, 1, 1), got {got}"


test_six_lots_by_hand()


def test_flag_nothing():
    got = confusion_counts(np.array([1, 0, 0, 1]), np.zeros(4, int))
    message = f"expected (2, 0, 2, 0) as Python ints, got {got}"
    assert got == (2, 0, 2, 0), message
    assert all(type(v) is int for v in got), message


test_flag_nothing()


def test_test_scores_kept():
    for name, s in (("s_lr", s_lr), ("s_rf", s_rf)):
        s = np.asarray(s)
        assert s.shape == (len(y_test),), f"{name}: one score per test lot"
        assert s.min() >= 0 and s.max() <= 1, f"{name}: a probability-like score in [0, 1]"


test_test_scores_kept()


print("all tests passed")

**Questions** (answer them in `practice_3_answers.md`, Task 2, with the numbers your code printed):

1. At the threshold 0.5, give the confusion matrix and the accuracy of each model on the test part, and the accuracy of "every lot passes".
2. What did each model do with the failed lots? Look at the largest score the forest gives to any test lot and explain where it comes from.

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

## Task 3. Precision, recall, F1 and the threshold  
*0.15 points: 0.08 for the code (tests), 0.07 for the answers*

$$\text{precision} = \frac{TP}{TP + FP} \qquad \text{recall} = \frac{TP}{TP + FN} \qquad F_1 = \frac{2PR}{P + R}$$

Two conventions, the same as in scikit-learn with `zero_division=0`: if nothing was flagged
($TP + FP = 0$), precision is 0; if $P + R = 0$, $F_1$ is 0. A model that flags nothing then
gets 0 everywhere, which is what it deserves.

$F_1$ is the harmonic mean of $P$ and $R$, so it sits near the smaller of the two:
$F_1 \le 2\min(P, R)$. A model cannot buy a good $F_1$ with a high recall alone.

To choose a threshold without touching the test part, every development lot needs a score from
a model that did not see it: **out-of-fold scores**,
`cross_val_predict(model, X_dev, y_dev, cv=CV, method="predict_proba")[:, 1]`.

**Write**

- `prf(tn, fp, fn, tp)`: the tuple `(precision, recall, f1)` of floats with the conventions above;
- `oof_lr`, `oof_rf`: the out-of-fold development scores of the two models, 1-D arrays in the
  order of `y_dev`. Task 6 needs them;
- `best_f1`: a dict `{"logistic": (t, f1), "forest": (t, f1)}`, the threshold on
  `np.linspace(0.01, 0.99, 99)` with the largest out-of-fold $F_1$ and that $F_1$. A lot is
  flagged when its score is $\ge t$; among equal $F_1$ take the smallest threshold.

In [ ]:
def prf(tn, fp, fn, tp):
    ...


oof_lr = ...
oof_rf = ...
best_f1 = {"logistic": ..., "forest": ...}

In [ ]:
# Tests for Task 3 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_known_counts():
    got = prf(tn=50, fp=10, fn=30, tp=10)
    assert got is not None, "prf returns (precision, recall, F1)"
    assert np.allclose(got, [0.5, 0.25, 1 / 3]), f"expected (0.5, 0.25, 0.333), got {got}"


test_known_counts()


def test_zero_conventions():
    nothing_flagged = tuple(prf(95, 0, 5, 0))
    only_good_flagged = tuple(prf(90, 5, 5, 0))
    assert nothing_flagged == (0.0, 0.0, 0.0), "nothing flagged: precision, recall, F1 are 0"
    assert only_good_flagged == (0.0, 0.0, 0.0), "flagged only good lots: all three are 0"


test_zero_conventions()


def test_out_of_fold_scores_kept():
    for name, s in (("oof_lr", oof_lr), ("oof_rf", oof_rf)):
        assert np.asarray(s).shape == (len(y_dev),), f"{name}: one score per development lot"
    assert set(best_f1) == {"logistic", "forest"}, "best_f1: (threshold, F1) per model"
    assert all(len(v) == 2 for v in best_f1.values()), "best_f1: (threshold, F1) per model"


test_out_of_fold_scores_kept()


print("all tests passed")

**Questions** (answer them in `practice_3_answers.md`, Task 3, with the numbers your code printed):

1. Precision, recall and F1 of both models on the test part at the threshold 0.5.
2. Plot precision, recall and F1 against the threshold on the out-of-fold scores, both models. Which threshold gives the best F1 for each, and what are precision and recall there?
3. Why is choosing the threshold on out-of-fold development scores allowed, and choosing it on the test scores not?

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

## Task 4. The ROC curve and its area  
*0.20 points: 0.10 for the code (tests), 0.10 for the answers*

Lower the threshold $t$ from above the largest score to below the smallest. At each distinct
score the rule "flag every lot with score $\ge t$" gives one point of the ROC curve:

$$\text{FPR}(t) = \frac{FP(t)}{n_{good}}, \qquad \text{TPR}(t) = \text{recall}(t) = \frac{TP(t)}{n_{fail}}.$$

The curve starts at $(0, 0)$ (nothing flagged) and ends at $(1, 1)$ (everything flagged).

ROC-AUC has a meaning without any curve. Take every pair (failed lot, good lot). ROC-AUC is the
share of pairs in which the failed lot got the **higher** score, with a tie counted as half:

$$\text{AUC} = \frac{\#\{s_{fail} > s_{good}\} + \tfrac{1}{2}\,\#\{s_{fail} = s_{good}\}}{n_{fail} \cdot n_{good}}$$

Why this is the area: passing one failed lot moves the curve up by $1/n_{fail}$, passing one good
lot moves it right by $1/n_{good}$. The column above a good lot's step has the height of the
share of failed lots already passed, those scored higher than this good lot. Summing the columns
counts the winning pairs. `pos[:, None]` against `neg[None, :]` gives a table with one cell per pair.

**Write**

- `auc_by_pairs(y, s)`: the formula above, a `float`;
- `roc_points(y, s)`: `(fpr, tpr)`, two 1-D arrays: first $(0, 0)$, then one point per distinct
  score, from the largest score down;
- `recall_at_fpr(y, s, max_fpr)`: the largest TPR among the points with FPR $\le$ `max_fpr`;
- `auc_test`: a dict `{"logistic": ..., "forest": ...}`, `auc_by_pairs` on the test part.

In [ ]:
def auc_by_pairs(y, s):
    ...


def roc_points(y, s):
    ...


def recall_at_fpr(y, s, max_fpr):
    ...


auc_test = {"logistic": ..., "forest": ...}

In [ ]:
# Tests for Task 4 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_four_lots_by_hand():
    y4, s4 = np.array([0, 0, 1, 1]), np.array([0.1, 0.4, 0.35, 0.8])
    got = auc_by_pairs(y4, s4)
    assert got is not None, "auc_by_pairs returns a number"
    assert np.isclose(got, 0.75), f"3 of the 4 pairs are ranked right, got {got}"


test_four_lots_by_hand()


def test_a_tie_counts_half():
    assert np.isclose(auc_by_pairs([0, 1], [0.5, 0.5]), 0.5), "one pair, a tie: AUC 0.5"
    assert np.isclose(auc_by_pairs([1, 0], [0.9, 0.2]), 1.0)


test_a_tie_counts_half()


def test_roc_points_by_hand():
    fpr, tpr = roc_points(np.array([0, 0, 1, 1]), np.array([0.1, 0.4, 0.35, 0.8]))
    assert np.allclose(fpr, [0, 0, 0.5, 0.5, 1]) and np.allclose(tpr, [0, 0.5, 0.5, 1, 1]), \
        f"thresholds 0.8, 0.4, 0.35, 0.1 after (0, 0); got fpr {fpr}, tpr {tpr}"


test_roc_points_by_hand()


def test_recall_at_a_false_alarm_budget():
    y4, s4 = np.array([0, 0, 1, 1]), np.array([0.1, 0.4, 0.35, 0.8])
    assert np.isclose(recall_at_fpr(y4, s4, 0.0), 0.5)
    assert np.isclose(recall_at_fpr(y4, s4, 0.5), 1.0)


test_recall_at_a_false_alarm_budget()


def test_auc_test_reported():
    assert set(auc_test) == {"logistic", "forest"}
    assert all(0 <= v <= 1 for v in auc_test.values())


test_auc_test_reported()


print("all tests passed")

**Questions** (answer them in `practice_3_answers.md`, Task 4, with the numbers your code printed):

1. ROC-AUC of both models on the test part, by your `auc_by_pairs` and by `roc_auc_score`. Plot both ROC curves with the diagonal of a coin.
2. The fab can afford to inspect 5 % of the good lots for nothing. What share of the failed lots does each model catch at a false-alarm rate of 5 %? And at 10 %?
3. Explain the forest's ROC-AUC in words, with the pairs definition: what exactly is that probability?

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

## Task 5. The precision-recall curve  
*0.20 points: 0.10 for the code (tests), 0.10 for the answers*

Lower the threshold as in Task 4 and record precision and recall at each distinct score. Average
precision is the area under the PR steps: each time recall grows, the width of the step times the
precision at its new value,

$$\text{AP} = \sum_n (R_n - R_{n-1})\, P_n, \qquad R_0 = 0,$$

with the points in the order of falling threshold. Read it as an average: each failed lot adds
$1/n_{fail}$ to recall when it gets flagged, so AP is the mean, over the failed lots, of the
precision at the moment each one is caught. A coin flags lots at random, its precision is the
failure rate everywhere, and so is its AP.

Precision counts false alarms against true ones, ROC counts them against all good lots. To see
what that means, **dilute** the test part: keep every failed lot once and repeat every good lot
$k$ times, with the same scores.

**Write**

- `pr_points(y, s)`: `(precision, recall)`, two 1-D arrays, one point per distinct score, from the
  largest score down (so recall never falls);
- `average_precision(y, s)`: the formula above, a `float`;
- `dilute(y, s, k)`: `(y_k, s_k)`, arrays in which each failed lot appears once and each good lot
  $k$ times, with its score;
- `ap_test`: a dict `{"logistic": ..., "forest": ...}`, `average_precision` on the test part;
- `diluted`: a dict `{"logistic": (auc, ap), "forest": (auc, ap)}`, ROC-AUC and AP after diluting
  the test part with $k = 10$.

In [ ]:
def pr_points(y, s):
    ...


def average_precision(y, s):
    ...


def dilute(y, s, k):
    ...


ap_test = {"logistic": ..., "forest": ...}
diluted = {"logistic": ..., "forest": ...}

In [ ]:
# Tests for Task 5 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_pr_points_by_hand():
    got = pr_points(np.array([0, 0, 1, 1]), np.array([0.1, 0.4, 0.35, 0.8]))
    assert got is not None, "pr_points returns (precision, recall)"
    prec, rec = got
    assert np.allclose(prec, [1, 0.5, 2 / 3, 0.5]) and np.allclose(rec, [0.5, 0.5, 1, 1]), \
        f"thresholds 0.8, 0.4, 0.35, 0.1; got precision {prec}, recall {rec}"


test_pr_points_by_hand()


def test_average_precision_by_hand():
    got = average_precision(np.array([0, 0, 1, 1]), np.array([0.1, 0.4, 0.35, 0.8]))
    assert got is not None, "average_precision returns a number"
    assert np.isclose(got, 0.5 * 1 + 0.5 * 2 / 3), f"expected 0.833, got {got}"


test_average_precision_by_hand()


def test_dilution_repeats_good_lots():
    got = dilute(np.array([1, 0, 0]), np.array([0.9, 0.2, 0.4]), 3)
    assert got is not None, "dilute returns (y_k, s_k)"
    yk, sk = map(np.asarray, got)
    assert len(yk) == len(sk) == 7 and yk.sum() == 1, "one failed lot and 2 x 3 good lots"
    assert sorted(sk[yk == 0]) == [0.2] * 3 + [0.4] * 3 and list(sk[yk == 1]) == [0.9]


test_dilution_repeats_good_lots()


def test_results_reported():
    assert set(ap_test) == {"logistic", "forest"}
    assert all(0 <= v <= 1 for v in ap_test.values())
    assert set(diluted) == {"logistic", "forest"}
    assert all(len(v) == 2 for v in diluted.values())


test_results_reported()


print("all tests passed")

**Questions** (answer them in `practice_3_answers.md`, Task 5, with the numbers your code printed):

1. AP of both models on the test part, with your function and with `average_precision_score`. What AP does a coin get here, and how many times better than the coin is each model? Plot both PR curves with the coin's level.
2. The dilution ×10: which metric moved, which did not, and why?
3. You report to the fab manager, who will read the number as "how good are the alarms". Which of the two metrics do you put in the report, and what do you write next to it?

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

## Task 6. The threshold from the cost of an error  
*0.25 points: 0.12 for the code (tests), 0.13 for the answers*

With a threshold $t$, every lot with score $\ge t$ is inspected. Then

$$\text{cost}(t) = C_{FP} \cdot FP(t) + C_{FN} \cdot FN(t)$$

$FP(t)$ falls and $FN(t)$ rises as $t$ grows, so the cost has a minimum somewhere between the
two trivial policies: $t = 0$ inspects every lot, $t$ above the largest score inspects none.
If the scores were calibrated probabilities, the minimum would sit at
$t^* = C_{FP} / (C_{FP} + C_{FN})$: inspect a lot when its chance of failing $p$ makes
$C_{FN}\,p > C_{FP}(1 - p)$ (In-class work 5, Part 5).

**Write**

- `cost_curve(y, s, thresholds, c_fp, c_fn)`: a 1-D array, the cost at each threshold;
- `cheapest_threshold(y, s, thresholds, c_fp, c_fn)`: the threshold with the lowest cost, the
  smallest one among equal costs;
- `chosen`: a dict `{"logistic": t, "forest": t}`, the cheapest threshold of each model on its
  **out-of-fold development scores** (Task 3), on `np.linspace(0, 1, 501)`;
- `test_cost`: a dict with the cost on the test part of four policies: `"inspect nothing"`,
  `"inspect every lot"`, `"logistic"` and `"forest"` (each model at its threshold in `chosen`).

In [ ]:
def cost_curve(y, s, thresholds, c_fp, c_fn):
    ...


def cheapest_threshold(y, s, thresholds, c_fp, c_fn):
    ...


chosen = {"logistic": ..., "forest": ...}
test_cost = {"inspect nothing": ..., "inspect every lot": ...,
             "logistic": ..., "forest": ...}

In [ ]:
# Tests for Task 6 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_six_lots_on_paper():
    y6 = np.array([0, 0, 0, 1, 0, 1])
    s6 = np.array([0.1, 0.4, 0.35, 0.8, 0.7, 0.3])
    got = cost_curve(y6, s6, [0.0, 0.32, 0.5, 0.9], c_fp=1, c_fn=20)
    assert got is not None, "cost_curve returns one cost per threshold"
    assert np.allclose(got, [4, 23, 21, 40]), f"expected [4, 23, 21, 40], got {got}"


test_six_lots_on_paper()


def test_cheapest_on_paper():
    y6 = np.array([0, 0, 0, 1, 0, 1])
    s6 = np.array([0.1, 0.4, 0.35, 0.8, 0.7, 0.3])
    grid = [0.0, 0.32, 0.5, 0.9]
    got = cheapest_threshold(y6, s6, grid, c_fp=1, c_fn=20)
    assert got is not None, "cheapest_threshold returns a threshold"
    assert np.isclose(got, 0.0), f"inspecting all six lots is cheapest here, got {got}"
    assert np.isclose(cheapest_threshold(y6, s6, grid, c_fp=10, c_fn=20), 0.5)


test_cheapest_on_paper()


def test_results_reported():
    assert set(chosen) == {"logistic", "forest"}
    assert all(0 <= v <= 1 for v in chosen.values())
    assert set(test_cost) == {"inspect nothing", "inspect every lot", "logistic", "forest"}
    assert all(v > 0 for v in test_cost.values())


test_results_reported()


print("all tests passed")

**Questions** (answer them in `practice_3_answers.md`, Task 6, with the numbers your code printed):

1. Plot the cost against the threshold on the out-of-fold development scores, both models, with the two fixed policies. Which threshold is cheapest for each, and at what cost?
2. Open the test part, once. One table: the cost of "inspect nothing", "inspect every lot", each model at 0.5, at $t^* = 1/21$ and at its development threshold, with a column "cost per 100 lots".
3. Which policy do you recommend to the fab? Choose it by the development costs, the same way you chose the thresholds; the test table only says what that choice costs. How much does it save on the test part against the best policy that needs no model?
4. Should $t^*$ work for these models? Are their scores calibrated probabilities (look at how the common cell trains them)? If a model looks cheap at $t^*$ on the test part anyway, would you trust that?

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

## Task 7. The same models, in time order  
*0.20 points: 0.08 for the code (tests), 0.12 for the answers*

So far the test lots were drawn at random from the three months. A real fab trains on the past
and runs on the future.

**Write**

- `time_split(times, frac=0.7)`: for a `pd.Series` of timestamps, `(past, future)`: two integer
  arrays of **positions**. Order the lots by time (equal times keep the order of the rows); the
  first `int(frac * n)` lots are the past, the rest the future;
- `s_time`: a dict `{"logistic": ..., "forest": ...}`: each model trained on the past of
  `time_split(lot_time)` (`X`, `y` of the common cell) and its scores on the future;
- `time_metrics`: a `pd.DataFrame` indexed by `"logistic"`, `"forest"` with columns `"ROC-AUC"`
  and `"AP"` on the future, from your functions of Tasks 4 and 5;
- `future_cost`: the cost on the future of the forest trained on the past, at your threshold
  `chosen["forest"]` from Task 6.

In [ ]:
def time_split(times, frac=0.7):
    ...


s_time = {"logistic": ..., "forest": ...}
time_metrics = ...
future_cost = ...

In [ ]:
# Tests for Task 7 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_split_by_time_not_by_row():
    days = ["2008-03-01", "2008-01-01", "2008-04-01", "2008-02-01", "2008-05-01"]
    t = pd.Series(pd.to_datetime(days))
    got = time_split(t, frac=0.6)
    assert got is not None, "time_split returns (past, future)"
    past, future = got
    message = f"past = the three earliest lots, got {past}, {future}"
    assert list(past) == [1, 3, 0] and list(future) == [2, 4], message


test_split_by_time_not_by_row()


def test_results_reported():
    assert set(s_time) == {"logistic", "forest"}
    assert isinstance(time_metrics, pd.DataFrame)
    assert {"ROC-AUC", "AP"} <= set(time_metrics.columns)
    assert future_cost is not None and future_cost > 0


test_results_reported()


print("all tests passed")

**Questions** (answer them in `practice_3_answers.md`, Task 7, with the numbers your code printed):

1. Is the order of the rows the order of the repaired time? Give the dates and the failed lots of the past and of the future.
2. ROC-AUC and AP on the future, next to the random-split numbers of Tasks 4–5, and the failure rate of each test part.
3. What does the forest policy of Task 6 cost on the future, against "inspect every lot" and "inspect nothing"?
4. Which of the two sets of numbers would you put in a report to the fab, and why? What could have changed between August and October?
5. Your conclusion in three or four sentences: the metric you would use to compare models on SECOM, the threshold you would install with what it costs per 100 lots, and how far you trust that number after this task.

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers